#### Steps taken:
1. Read bronze sprints data.
2. Keep only columns required for analytics, will drop the url column.
3. Standardize the column name using snake_case (ex: driverID -> driver_id, positionText -> finish_position_text etc.)
4. Rename columns to make it more meaningful.
5. Filter out rows where season, round, constructor_id, driver_id that is null (business key validation)
5. Remove duplicate records.
6. Transform values of columns to title case (ex: race_name).
7. Write the transform data to silver results table.

In [0]:
%run ../environment_config

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.sprints"
silver_table = f"{catalog}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = spark.read.table(bronze_table)\
                  .select("*").drop("url")\
                  .withColumnsRenamed({
                         "constructorId" : "constructor_id",
                         "date" : "race_date",
                         "driverID" : "driver_id",
                         "raceName" : "race_name",
                         "grid" : "grid_position",
                         "laps" : "completed_laps",
                         "number" : "car_number",
                         "position" : "final_position",
                         "positionText" : "final_position_text"
                  })

In [0]:
sprints_valid_df = sprints_df.filter(
                    F.col("season").isNotNull() &
                    F.col("round").isNotNull() &
                    F.col("constructor_id").isNotNull() &
                    F.col("driver_id").isNotNull() 
                     )

In [0]:
display(sprints_df.count() - sprints_valid_df.count())

In [0]:
display(sprints_valid_df.count())

In [0]:
sprints_distinct_df = sprints_valid_df.dropDuplicates(
                         ["season", "round", "driver_id", "constructor_id"]
                     )

In [0]:
display(sprints_distinct_df.count())

In [0]:
display(sprints_valid_df.count() - sprints_distinct_df.count())

In [0]:
sprints_final_df = sprints_distinct_df.withColumn(
                                "race_name", F.initcap(F.col("race_name"))
)

In [0]:
(
    sprints_final_df.write
                    .format("delta")
                    .mode("overwrite")
                    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))